# Visualisation & story — European commercial airspace

Reads the result parquets produced by `04_spark_etl.ipynb` (`../data/processed/`) and turns them into charts. **No Kafka / Spark needed** — purely on the stored results.

- **Q1** Dominance — top airlines & carrier nations
- **Q2** Geography — flight-density map (+ whose airspace)
- **Q3** Busiest airports
- **Q4a** Departures vs arrivals
- **Extras** — altitude distribution & heading flow-field

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = "../data/processed"

by_airline    = pd.read_parquet(f"{DATA_DIR}/dominance_by_airline.parquet")
by_country    = pd.read_parquet(f"{DATA_DIR}/dominance_by_country.parquet")
density       = pd.read_parquet(f"{DATA_DIR}/density_grid.parquet")
busiest       = pd.read_parquet(f"{DATA_DIR}/busiest_airports.parquet")
departures    = pd.read_parquet(f"{DATA_DIR}/top_departures.parquet")
arrivals      = pd.read_parquet(f"{DATA_DIR}/top_arrivals.parquet")
altitude_hist = pd.read_parquet(f"{DATA_DIR}/altitude_hist.parquet")
flow_field    = pd.read_parquet(f"{DATA_DIR}/flow_field.parquet")
print("loaded all result parquets")

## Q1 — Dominance: who rules the European sky?

In [ ]:
top = by_airline.head(12).iloc[::-1]
plt.figure(figsize=(9, 5))
plt.barh(top["airline"], top["aircraft"], color="#2b8cbe")
plt.xlabel("distinct aircraft aloft")
plt.title("Top airlines over Europe (commercial)")
plt.tight_layout()
plt.show()

In [ ]:
topc = by_country.head(12).iloc[::-1]
plt.figure(figsize=(9, 5))
plt.barh(topc["origin_country"], topc["aircraft"], color="#41ab5d")
plt.xlabel("distinct aircraft aloft")
plt.title("Top carrier nations over Europe (by operator country)")
plt.tight_layout()
plt.show()

## Q2 — Geography: the shape of Europe's air traffic
Flight density binned into hexagons (log colour) with **country borders** on top. Bright hexes = busiest hubs/corridors.

In [ ]:
import requests

# Country borders overlay — lightweight: a plain GeoJSON drawn with matplotlib
# (no cartopy/geopandas). Natural Earth 1:110m admin-0 countries.
BORDERS_URL = "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_110m_admin_0_countries.geojson"
_geo_cache = {}

def plot_borders(ax, color="white", lw=0.5):
    try:
        geo = _geo_cache.get("geo") or requests.get(BORDERS_URL, timeout=60).json()
        _geo_cache["geo"] = geo
    except Exception as e:
        print("borders unavailable:", e)
        return
    for feat in geo["features"]:
        g = feat["geometry"]
        polys = g["coordinates"] if g["type"] == "MultiPolygon" else [g["coordinates"]]
        for poly in polys:
            for ring in poly:
                ax.plot([c[0] for c in ring], [c[1] for c in ring],
                        color=color, lw=lw, zorder=3)

fig, ax = plt.subplots(figsize=(11, 8))
hb = ax.hexbin(density["lon_bin"], density["lat_bin"], C=density["flights"],
               reduce_C_function=np.sum, gridsize=55, cmap="inferno",
               bins="log", mincnt=1, zorder=1)
plot_borders(ax)
fig.colorbar(hb, label="flight records (log scale)")
ax.set_xlim(-25, 50)
ax.set_ylim(34, 72)
ax.set_aspect(1 / np.cos(np.radians(53)))
ax.set_title("European commercial air-traffic density")
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
fig.tight_layout()
plt.show()

### Whose airspace is busiest?
Reverse-geocode each grid cell to a country and sum the flights. Needs `reverse_geocoder` (offline): `pip install reverse_geocoder`.

In [ ]:
try:
    import reverse_geocoder as rg
    coords = list(zip(density["lat_bin"].tolist(), density["lon_bin"].tolist()))
    hits = rg.search(coords, mode=1)
    density["cc"] = [h["cc"] for h in hits]
    airspace = (density.groupby("cc")["flights"].sum()
                .sort_values(ascending=False).head(12).iloc[::-1])
    plt.figure(figsize=(9, 5))
    plt.barh(airspace.index, airspace.values, color="#d7301f")
    plt.xlabel("flight records over the country's territory")
    plt.title("Busiest airspace by country (overflights)")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("reverse_geocoder not installed -> run: pip install reverse_geocoder")

## Q3 — Busiest airports (reconstructed from live positions)

In [ ]:
topp = busiest.head(15).iloc[::-1]
plt.figure(figsize=(9, 6))
plt.barh(topp["airport_name"], topp["aircraft"], color="#6a51a3")
plt.xlabel("distinct aircraft near the airport")
plt.title("Busiest European airports (inferred from live traffic)")
plt.tight_layout()
plt.show()

## Q4a — Departures vs arrivals

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
d = departures.head(10).iloc[::-1]
axes[0].barh(d["airport_name"], d["departures"], color="#3182bd")
axes[0].set_title("Top departure airports")
axes[0].set_xlabel("distinct departing aircraft")
a = arrivals.head(10).iloc[::-1]
axes[1].barh(a["airport_name"], a["arrivals"], color="#e6550d")
axes[1].set_title("Top arrival airports")
axes[1].set_xlabel("distinct arriving aircraft")
plt.tight_layout()
plt.show()

## Extra 1 — Altitude distribution (flight levels)
Where does Europe fly? The peaks reveal the discrete cruising **flight levels**.

In [ ]:
ah = altitude_hist.sort_values("alt_bin")
plt.figure(figsize=(7, 7))
plt.barh(ah["alt_bin"] / 1000, ah["flights"], height=0.45, color="#2c7fb8")
plt.ylabel("altitude (km)")
plt.xlabel("flight records")
plt.title("Where Europe flies: altitude distribution")
plt.tight_layout()
plt.show()

## Extra 2 — Flow field: how traffic moves
Each arrow is the **dominant heading** in a ~1° cell (mean of `true_track`), coloured by traffic volume. Shows the directional flow across Europe.

In [ ]:
flow = flow_field[flow_field["flights"] >= 5]   # declutter: only well-sampled cells
fig, ax = plt.subplots(figsize=(11, 9))
plot_borders(ax, color="0.5")                   # grey borders on white background
q = ax.quiver(flow["lon_c"], flow["lat_c"], flow["mean_u"], flow["mean_v"], flow["flights"],
              cmap="viridis", angles="uv", scale=30, width=0.003, zorder=2)
fig.colorbar(q, label="flights per cell")
ax.set_xlim(-25, 50)
ax.set_ylim(34, 72)
ax.set_aspect(1 / np.cos(np.radians(53)))
ax.set_title("Flow field — dominant heading per ~1° cell")
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
fig.tight_layout()
plt.show()

## Story / takeaways
- **Who owns the European sky?** A few low-cost & legacy carriers dominate — Ryanair far ahead, then Turkish, easyJet, Lufthansa, Air France…
- **Where?** The density map traces the classic core corridors (UK – Benelux – Germany – N. Italy) against the country outlines.
- **Busiest hubs** reconstructed from raw positions match the real ranking (Istanbul, Heathrow, Frankfurt, Schiphol, CDG…) — live ADS-B data alone recovers reality.
- **Altitude** shows the layered structure of airspace (discrete flight levels); the **flow field** shows the dominant directions of travel.

_Numbers reflect the snapshot window captured in Kafka at ETL time; rerunning `04` with more accumulated data sharpens them._